# Model-input boundary validation

Approved inputs are FULL v1 and MASKED v2 opinion text, with separate targets
from the frozen split manifest. Only `text` goes to a tokenizer/vectorizer;
`case_id` is bookkeeping and `target` is separate supervision/scoring data.
No annotation summaries, findings, favors or selected quotations enter the loader.
This does not remove reasoning or quotations naturally present in the opinion.

The 114 saved texts share IDs; the loader returns the 113 assigned cases
(81 training, 16 development, 16 test), leaving Sodhi pending. Targets remain
provisional. See `MODEL_INPUT_HANDOFF.md` for the exact input contract and limits.


In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside legal-judgment.")
MVP = ROOT / "annotation/submissions/becca/MVP"
sys.path.insert(0, str(MVP / "preprocessing"))
import model_inputs
import verify_model_inputs
_, _, _, report = model_inputs.read_validated(MVP)
print("Selection:", report["selection_count"])
print("Partitions:", report["partition_counts"])
print("Pending:", report["pending_case_ids"])
print("Target status:", report["target_review_status_counts"])
print("Near-empty texts:", report["near_empty_case_ids_by_variant"])


Selection: 114
Partitions: {'train': 81, 'development': 16, 'test': 16}
Pending: ['NY-4655053']
Target status: {'pending_hayden_review': 114}
Near-empty texts: {'FULL': [], 'MASKED': []}


## Verify the feature/target boundary

Mutation checks introduce prohibited fields, invalid IDs/targets, empty text and
family overlap in memory only; saved datasets are never changed. Both variants
must return identical case IDs and labels, regardless of export row order.
The loader accepts text strings only and returns targets separately.


In [2]:
checks = verify_model_inputs.check(MVP)
print(json.dumps(checks, indent=2))
for partition in model_inputs.PARTITIONS:
    full = model_inputs.load_model_batch(MVP, "FULL", partition)
    masked = model_inputs.load_model_batch(MVP, "MASKED", partition)
    assert full.case_ids == masked.case_ids and full.targets == masked.targets
    print(partition, len(full.case_ids), "matching IDs and targets; only texts enter feature construction")


{
  "checks_passed": 17,
  "checks": [
    "reject_label",
    "reject_target",
    "reject_favors",
    "reject_legal_findings",
    "reject_factual_summary",
    "reject_supporting_quotations",
    "reject_metadata",
    "duplicate_id",
    "missing_variant_case",
    "missing_target",
    "boolean_target",
    "empty_text",
    "nested_annotation_in_text",
    "family_overlap",
    "join_by_id_not_file_order",
    "identical_targets_across_variants",
    "pending_cases_excluded"
  ]
}
train 81 matching IDs and targets; only texts enter feature construction
development 16 matching IDs and targets; only texts enter feature construction
test 16 matching IDs and targets; only texts enter feature construction


## Save validation provenance

Record the exact inputs, validator and check-script hashes. A later source,
label, or preprocessing change requires a new version and affected experiments
rerun. This report does not establish human-gold labels or exhaustive masking.
No tokenizer, classifier, or external service is run.


In [3]:
report["checks"] = checks
report["validator_sha256"] = model_inputs.sha256(MVP / "preprocessing/model_inputs.py")
report["check_script_sha256"] = model_inputs.sha256(MVP / "preprocessing/verify_model_inputs.py")
path = MVP / "data/model_input_contract_v1/validation.json"
data = (json.dumps(report, indent=2) + "\n").encode()
if path.exists() and path.read_bytes() != data:
    raise ValueError("Existing validation version differs; create a new contract version.")
path.parent.mkdir(parents=True, exist_ok=True)
if not path.exists():
    path.write_bytes(data)
print("Saved:", path.relative_to(MVP))


Saved: data/model_input_contract_v1/validation.json
